# 05 統合シミュレーション

家事・経済的負担・コミュニケーション・意思決定戦略を統合し、**長期的な関係継続**を比べます（RQ3〜RQ6）。

> **研究課題**：毎年の平均満足度を最大化する戦略は、長期的な関係継続率も最大化するか？
> 結果はこの Notebook を実行して確かめてください。**結果はモデルの仮定に依存します。**

> **教育用の架空データ・架空モデル**です。Partner A・B は実在の人物ではなく、性別や役割を意味しません。満足度関数や関係解消確率の係数は架空の値で、実際の離婚率や特定の関係の将来を予測するものではありません。

## 準備：実験環境（最初に必ず実行）

F4 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（F4 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | F4 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（F4 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                                 # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "f4-partnership-simulation"     # リポジトリの中の F4 フォルダ（ルートが F4 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_f4_root(p):
    return (Path(p) / "household_model.py").exists() and (Path(p) / "relationship_dynamics.py").exists()


def find_f4_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または F4 のルート
        if is_f4_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_f4_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("F4 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/f4_upload")
    for hit in glob.glob("/content/f4_upload/**/household_model.py", recursive=True):
        if is_f4_root(Path(hit).parent):
            return Path(hit).parent
    return None


F4_ROOT = find_f4_root()
if F4_ROOT is None:
    raise RuntimeError("F4 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(F4_ROOT)
if str(F4_ROOT) not in sys.path:
    sys.path.insert(0, str(F4_ROOT))
if not (F4_ROOT / "textbook" / "max.py").exists():
    print("⚠️ textbook/ がありません。decision_strategies.py などの import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import partners as pt
import household_model as hm
import decision_strategies as ds
import relationship_dynamics as rd
import evaluation as ev
import partnership_charts as pc

pc.setup_style()                       # 文字を大きくするなら pc.setup_style(1.3)
scenarios = pt.load_scenarios()        # シナリオ A〜D（すべて架空）
params = rd.load_parameters()          # モデルのパラメータ（すべて架空）
events = pt.load_stress_events()       # ストレスイベントの組
SC = pt.scenario_by_id(scenarios, "A") # 基準にする初期条件（Scenario A）
P, HH = SC["partners"], SC["household"]
COND0 = dict(HH, stress=params["base_stress"])   # ストレスイベントのない年の生活条件
CANDS = ds.candidate_grid(params["share_step"])  # 分担の候補（全戦略で共通）
SEEDS = range(5)                       # 乱数シード
REPLICATES = 200                       # シードごとの反復回数
YEARS = params["simulation_years"]

SAVE_FIGURES = False                   # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", pc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    data = [{h: ("-" if r.get(k) is None else f.format(r[k])) for k, h, f in columns} for r in rows]
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(data))
    except ImportError:
        print(ev.format_table(rows, columns))


print("F4_ROOT:", F4_ROOT)
print("シナリオ:", [s["label"] for s in scenarios])
print("モデルのパラメータ（架空）:", {k: params[k] for k in ("alpha", "baseline_hazard", "beta", "gamma",
                                                       "base_stress", "fairness_lambda", "share_step", "simulation_years")})

## 実験条件

- 初期条件：Scenario A のパートナー・家計（全戦略で同じ）
- ストレスイベント：`combined`（Year 5：A の所得 −20%、Year 8：家事の総時間 +10 時間/週）
- 乱数：5 シード × 200 回（**共通乱数**：戦略が違っても同じ乱数列）
- 満足度・公平性は「関係が続いた場合」の 20 年平均、継続率はシードごとの継続率の平均 ± シード間の標準偏差

In [ ]:
EVENTS = events["combined"]
res = ev.compare_strategies(SC, params, EVENTS, ds.STRATEGIES, REPLICATES, SEEDS)
table([{"name": ds.LABELS[s], **res[s]} for s in ds.STRATEGIES],
      [("name", "strategy", "{}"), ("first_year_mean_utility", "year-1 mean", "{:.3f}"),
       ("mean_satisfaction", "mean sat.", "{:.3f}"), ("min_satisfaction", "min sat.", "{:.3f}"),
       ("fairness", "fairness", "{:.3f}"), ("continuation_5", "5Y", "{:.1%}"), ("continuation_10", "10Y", "{:.1%}"),
       ("continuation_20", "20Y", "{:.1%}"), ("continuation_20_sd", "20Y sd", "{:.1%}")])

## 1〜3. 戦略別の平均満足度・最低満足度・公平性

In [ ]:
#@title 戦略別の平均満足度・最低満足度・公平性
show(pc.plot_strategy_metrics(res, [("mean_satisfaction", "mean satisfaction"), ("min_satisfaction", "min satisfaction"),
                                    ("fairness", "fairness index")],
                              "Mean, minimum satisfaction and fairness (20-year average, model)"), "05_metrics")

## 4. 戦略別の 5・10・20 年継続率

誤差棒 = シード間の標準偏差。**差が誤差棒と同じくらいなら、優劣は判断できません。**

In [ ]:
#@title 戦略別の継続率
show(pc.plot_checkpoints(res, "Continuation rate by strategy (model, mean ± sd over seeds)"), "05_checkpoints")

## 5. ストレス条件別の継続率（RQ5）

In [ ]:
#@title ストレス条件別の継続率
STRESS_SETS = ["none", "income_drop", "work_increase", "housework_increase", "expense_increase", "combined"]
stress_rows = []
for name in STRESS_SETS:
    r = ev.compare_strategies(SC, params, events[name], ds.STRATEGIES, REPLICATES, SEEDS)
    stress_rows.append({"name": name.replace("_", "\n"), **r})
show(pc.plot_condition_bars(stress_rows, "continuation_10", "10-year continuation by stress condition (model)",
                            "10-year continuation rate"), "05_stress_conditions")

## コミュニケーション条件別（RQ3）

2 人のコミュニケーション指標を 0.2・0.5・0.8 にしたときの 10 年継続率です。

In [ ]:
#@title コミュニケーション条件別の継続率
comm_rows = []
for level in [0.2, 0.5, 0.8]:
    r = ev.compare_strategies(pt.with_communication(SC, level), params, EVENTS, ds.STRATEGIES, REPLICATES, SEEDS)
    comm_rows.append({"name": f"communication {level}", **r})
show(pc.plot_condition_bars(comm_rows, "continuation_10", "10-year continuation by communication level (model)",
                            "10-year continuation rate"), "05_communication")

## 6. パラメータ感度分析

1 つずつパラメータを変えて、10 年継続率がどう変わるかを見ます。**戦略の順位が入れ替わる条件**に注目してください。

In [ ]:
#@title パラメータ感度分析
CHANGES = [("baseline", {}), ("alpha 0.1", {"alpha": 0.1}), ("alpha 0.6", {"alpha": 0.6}),
           ("hazard 0.03", {"baseline_hazard": 0.03}), ("beta 0.04", {"beta": 0.04}),
           ("min weight 0", {"relationship_min_weight": 0.0}), ("w_fairness 0.6", {"w_fairness": 0.6})]
sens_rows = []
for name, ch in CHANGES:
    r = ev.compare_strategies(SC, rd.with_changes(params, **ch), EVENTS, ds.STRATEGIES, REPLICATES, SEEDS)
    sens_rows.append({"name": name.replace(" ", "\n"), **r})
show(pc.plot_condition_bars(sens_rows, "continuation_10", "Sensitivity: 10-year continuation (model)",
                            "10-year continuation rate"), "05_sensitivity")
for row in sens_rows:
    order = ds.rank_strategies({s: row[s]["continuation_10"] for s in ds.STRATEGIES})
    print(f"{row['name'].replace(chr(10), ' '):>15}: " + " > ".join(ds.LABELS[s] for s, _v in order))

## RQ6：短期の満足度と長期の継続

横軸 = 1 年目の平均満足度、縦軸 = 20 年継続率です。

In [ ]:
#@title 短期の満足度と長期の継続
show(pc.plot_short_vs_long(res, "Short-term satisfaction vs long-term continuation (model)"), "05_short_vs_long")
first = ds.rank_strategies({s: res[s]["first_year_mean_utility"] for s in ds.STRATEGIES})
long = ds.rank_strategies({s: res[s]["continuation_20"] for s in ds.STRATEGIES})
print("1 年目の平均満足度の順位:", " > ".join(ds.LABELS[s] for s, _v in first))
print("20 年継続率の順位        :", " > ".join(ds.LABELS[s] for s, _v in long))

## 学生向け TODO

In [ ]:
# TODO 1：シード数・反復回数を増やして、結論が変わらないか確かめる
MY_SEEDS, MY_REPLICATES = range(10), 500
# TODO 2：別の初期条件（Scenario B〜D のパートナー・分担ルール）で比べる
my_sc = pt.scenario_by_id(scenarios, "C")
# TODO 3：communication_buffer = 0 にして、RQ3 の結果が変わるか確かめる
my_params = rd.with_changes(params, communication_buffer=0.0)
r = ev.compare_strategies(my_sc, my_params, EVENTS, ds.STRATEGIES, MY_REPLICATES, MY_SEEDS)
for s in ds.STRATEGIES:
    print(f"{ds.LABELS[s]:15}: 10 年 {r[s]['continuation_10']:.1%} ± {r[s]['continuation_10_sd']:.1%}"
          f"  20 年 {r[s]['continuation_20']:.1%}  最低満足度 {r[s]['min_satisfaction']:.3f}")
# TODO 4：4 つの指標（平均・最低・公平性・継続率）をまとめて評価する方法を考える（Advanced D）

## 考察（チームで記入）

- **TODO**：毎年の平均満足度を最大化する戦略（Greedy）は、長期的な継続率も最大化したか（RQ6）
- **TODO**：最低満足度・公平性と継続率の関係は、モデルのどの式から来ているか（$R(t)$ の $m$、$\beta$ など）
- **TODO**：どのストレスイベントの影響が大きかったか。戦略によって違ったか（RQ5）
- **TODO**：コミュニケーションは負担の不均衡の影響を和らげたか。それは `communication_buffer` の仮定によるものか（RQ3）
- **TODO**：感度分析で順位が入れ替わった条件はあるか。結論はどの仮定の下で成り立つか

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第2章 最大値、第3章 線形探索、第6章 単純挿入ソート）
- F4 リポジトリの `README.md`（モデルの仮定・計算量・モデルの限界と倫理的配慮）
- データ：`data/sample_partnerships.json`（この教材のための**架空のデータ**）